# IMPPAT anti-target neighborhood

Full reproduction of the current manuscript (`manuscript.md`).
Open this notebook from the `imppat_overlap` folder and run it from the top.

The calculation stays in `src/`. This notebook runs those scripts in manuscript order and then shows the tables and figures.

1. Similarity, cutoffs, Table 1, and the unblinded PXR compounds (`paper_analysis.py`)
2. Descriptor separation (`separate_close_far.py`)
3. Plant enrichment, the potency filter, and Supplementary Table S3 (`plant_enrichment.py`, `potent_split.py`)
4. Robustness checks (`revision_checks.py`): reference-relative coverage, fingerprint mean versus minimum, CYP isoforms, strongest neighbor at 0.80, ubiquitous-compound caps, curation, and unmapped names
5. Figures 1-5 (`figures.py`; Figure 3D uses `relative_coverage.csv`)

Requires Python 3.11 with RDKit, pandas, numpy, scikit-learn, and matplotlib.
The similarity search, the descriptor search, and the rescore inside the robustness checks are the slow steps.


## 1. Setup


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

def find_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "imppat_overlap", *cwd.parents]
    for candidate in candidates:
        if (candidate / "src" / "paper_analysis.py").exists() and (candidate / "input" / "imppat.csv").exists():
            return candidate
    raise SystemExit("Open this notebook from the imppat_overlap folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT / "src"))
PAPER = ROOT / "results" / "paper"
INPUT = ROOT / "input"
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 160)

def show(title, obj):
    display(Markdown("**" + title + "**"))
    display(obj)

required = [
    "hERG_Dataset.csv",
    "cyp-challenge-TRAIN_inhibition.csv",
    "pxr-challenge_TRAIN.csv",
    "pxr-challenge_TEST_PHASE_1_UNBLINDED.csv",
    "pxr-challenge_TEST_PHASE_2_UNBLINDED.csv",
    "imppat.csv",
]
missing = [name for name in required if not (INPUT / name).exists()]
n_sdf = len(list((INPUT / "sdf_files").glob("*.sdf")))
print("input files")
for name in required:
    path = INPUT / name
    print(f"  {name}: {path.stat().st_size / 1e6:.2f} MB" if path.exists() else f"  MISSING {name}")
print(f"  sdf_files: {n_sdf} structures")
failed = INPUT / "failed_pubchem_downloads.csv"
print(f"  failed_pubchem_downloads.csv: {'present' if failed.exists() else 'MISSING'}")
if missing or n_sdf == 0:
    raise SystemExit("Input folder is incomplete.")
if not failed.exists():
    print("Robustness checks need input/failed_pubchem_downloads.csv for the unmapped-name section.")


## 2. Similarity, cutoffs, and the unblinded PXR test

Manuscript sections 2.3, 3.1, and 3.2. For every phytochemical, similarity is the lower of the Morgan and atom-pair Tanimoto values to the same measured neighbor. Close means that consensus value is at least 0.80. The 513 unblinded PXR compounds are scored separately and are not added to the training reference set.

Writes `imppat_similarity.csv`, `cutoff_fractions.csv`, `descriptor_differences.csv`, `table1_summary.csv`, `close_compounds.csv`, `within_set_similarity.csv`, and `pxr_unblinded_similarity.csv`.


In [ ]:
import paper_analysis
paper_analysis.main()

summary = pd.read_csv(PAPER / "table1_summary.csv")
robust = pd.read_csv(PAPER / "cutoff_fractions.csv")
effect = pd.read_csv(PAPER / "descriptor_differences.csv")
similarity = pd.read_csv(PAPER / "imppat_similarity.csv")
unblinded = pd.read_csv(PAPER / "pxr_unblinded_similarity.csv")
show("Table 1", summary.round(4))
at_80 = robust[robust.cutoff == 0.80][["anti_target", "rule", "n", "fraction", "ci_low", "ci_high"]]
show("Counts at similarity 0.80", at_80.round(4))
effect = effect.copy()
effect["interval_excludes_0"] = (effect.ci_low > 0) | (effect.ci_high < 0)
show("Close minus far", effect.round(3))
print(f"structures {len(similarity)}")
print(f"unblinded PXR rows {len(unblinded)}; max consensus {unblinded.consensus.max():.3f}")


## 3. Descriptor separation

Manuscript section 3.4. Each anti-target is its own comparison. Logistic regression is scored by five-fold cross-validation on single descriptors, on pairs and triples of the eight strongest singles, and on all 24 descriptors. The winning set is compared with 200 label permutations.


In [ ]:
import separate_close_far
separate_close_far.main()

separation = pd.read_csv(PAPER / "separation_summary.csv")
show("Descriptor separation", separation.round(3))
for name in ("hERG", "CYP", "PXR"):
    combos = pd.read_csv(PAPER / f"descriptor_combinations_{name}.csv")
    show(name + ": highest combinations", combos.head(8).round(3))


## 4. Plant enrichment and potent neighbors

Manuscript sections 3.5 and 3.6. A plant's score is how many constituents are close at 0.80. The null keeps that plant's constituent count fixed. `restricted` drops compounds recorded from more than 20 plants. The potency pass keeps neighbors with pIC50 or pEC50 of at least 5, then repeats the plant test. Supplementary Table S3 lists those potent close phytochemicals.


In [ ]:
import plant_enrichment
import potent_split

plant_enrichment.main()
potent_split.main()

enrichment = pd.read_csv(PAPER / "plant_enrichment.csv")
count_rows = []
for (name, label), subset in enrichment.groupby(["anti_target", "compound_set"], sort=False):
    count_rows.append({
        "anti_target": name,
        "compound_set": label,
        "plants_with_a_hit": len(subset),
        "q_below_0.05": int((subset.q_value < 0.05).sum()),
    })
show("Plant enrichment summary", pd.DataFrame(count_rows))
restricted = enrichment[(enrichment.compound_set == "restricted") & (enrichment.q_value < 0.05)]
show("Restricted test, q < 0.05", restricted.sort_values(["anti_target", "q_value", "plant"]).round(4))

potent = pd.read_csv(PAPER / "plant_enrichment_potent.csv")
named = potent[(potent.q_value < 0.05) & (potent.n_close >= 2)]
show("Potent neighbors, q < 0.05 and at least two constituents", named.sort_values(["anti_target", "q_value", "plant"]).round(4))
supplement = pd.read_csv(ROOT / "supplementary_table_S3_compounds.csv")
show("Supplementary Table S3", supplement.groupby("anti_target").size().rename("rows").to_frame())


## 5. Robustness checks

Manuscript sections 2.2, 3.3, 3.7, and 3.8, from `revision_checks.py`.

- Where a typical IMPPAT consensus similarity sits in each reference set's own nearest-neighbor distribution, and how many phytochemicals reach that set's median.
- Minimum versus mean of the two fingerprints.
- CYP1A2, CYP2C9, CYP2D6, and CYP3A4 counted separately.
- Potency of the nearest neighbor versus the strongest neighbor that is still at consensus similarity 0.80 or above.
- Plant caps of 10, 20, and 50.
- Curation of the 206 close structures into credible phytochemicals, xenobiotics, contaminants, and uncertain records.
- Names that failed structure retrieval.

The rescore reads the SDF files again and is slow. Figure 3D uses `relative_coverage.csv` from this step. `unmapped_bias` reads `input/failed_pubchem_downloads.csv`.


In [ ]:
import revision_checks
revision_checks.main()

show("Coverage relative to each reference set", pd.read_csv(PAPER / "relative_coverage.csv").round(4))
show("Minimum versus mean of the two fingerprints", pd.read_csv(PAPER / "fingerprint_aggregation.csv"))
show("CYP isoforms", pd.read_csv(PAPER / "cyp_isoform_coverage.csv").round(4))
show("Nearest neighbor versus strongest neighbor at 0.80", pd.read_csv(PAPER / "neighbor_potency_rules.csv"))
show("Ubiquitous-compound caps", pd.read_csv(PAPER / "ubiquitous_sensitivity.csv"))
show("Close set after removing xenobiotics and contaminants", pd.read_csv(PAPER / "coverage_after_curation.csv"))
curation = pd.read_csv(PAPER / "close_compound_curation.csv")
show("Curation classes", curation.curation.value_counts().rename("n").to_frame())
show("Unmapped names", pd.read_csv(PAPER / "unmapped_name_summary.csv").round(3))


## 6. Figures 1-5

- **Figure 1.** Diversity of the measured sets.
- **Figure 2.** IMPPAT consensus similarity, the close-set intersections, and the unblinded PXR compounds.
- **Figure 3.** Cutoff and fingerprint rule. Panel D compares consensus similarity of at least 0.80 with each reference set's own median nearest-neighbor similarity.
- **Figure 4.** Close versus far descriptors, and the cross-validated separation.
- **Figure 5.** Plant enrichment before and after dropping compounds recorded from more than 20 plants.


In [ ]:
import figures
figures.main()

for name in (
    "figure1_within_set_diversity.png",
    "figure2_imppat_similarity.png",
    "figure3_cutoff_robustness.png",
    "figure4_close_vs_far.png",
    "figure5_plant_enrichment.png",
):
    path = PAPER / name
    display(Markdown("### " + name))
    display(Image(filename=str(path)))


## 7. Checks against the manuscript

Consensus similarity of at least 0.80 gives 137, 45, and 69 phytochemicals. Morgan alone gives 212, 85, and 147. The mean of the two fingerprints gives 188, 67, and 108. Reaching each reference set's own median nearest-neighbor similarity gives 142, 859, and 1,643.

Of the consensus-close phytochemicals, 51, 16, and 11 have a nearest neighbor at potency 5 or higher. Using the strongest neighbor at consensus similarity of at least 0.80 raises the CYP count from 16 to 19 and leaves hERG and PXR at 51 and 11.

Restricting plant enrichment to compounds recorded from at most 20 plants leaves 12, 6, and 4 plants at q < 0.05. Caps of 10 and 50 leave 8, 3, 7 and 16, 6, 5. Removing xenobiotics and contaminants lowers the close counts to 113, 40, and 65. The closest IMPPAT structure to the unblinded PXR compounds is below 0.50.


In [ ]:
similarity = pd.read_csv(PAPER / "imppat_similarity.csv")
unblinded = pd.read_csv(PAPER / "pxr_unblinded_similarity.csv")
robust = pd.read_csv(PAPER / "cutoff_fractions.csv")
summary = pd.read_csv(PAPER / "table1_summary.csv")
separation = pd.read_csv(PAPER / "separation_summary.csv")
supplement = pd.read_csv(ROOT / "supplementary_table_S3_compounds.csv")
relative = pd.read_csv(PAPER / "relative_coverage.csv")
aggregation = pd.read_csv(PAPER / "fingerprint_aggregation.csv")
isoforms = pd.read_csv(PAPER / "cyp_isoform_coverage.csv")
potency_rules = pd.read_csv(PAPER / "neighbor_potency_rules.csv")
ubiquitous = pd.read_csv(PAPER / "ubiquitous_sensitivity.csv")
curated = pd.read_csv(PAPER / "coverage_after_curation.csv")
classes = pd.read_csv(PAPER / "close_compound_curation.csv")
enrichment = pd.read_csv(PAPER / "plant_enrichment.csv")
unmapped = pd.read_csv(PAPER / "unmapped_name_summary.csv")

def count_at(name, rule, cutoff=0.80):
    row = robust[(robust.anti_target == name) & (robust.rule == rule) & (robust.cutoff == cutoff)]
    return int(row["n"].iloc[0])

consensus = {name: count_at(name, "consensus") for name in ("hERG", "CYP", "PXR")}
morgan = {name: count_at(name, "morgan") for name in ("hERG", "CYP", "PXR")}
intersections = summary.set_index("group")["n_close_consensus_0.80"].astype(int).to_dict()
plant_counts = {}
for name in ("hERG", "CYP", "PXR"):
    for label in ("all", "restricted"):
        subset = enrichment[(enrichment.anti_target == name) & (enrichment.compound_set == label)]
        plant_counts[(name, label)] = int((subset.q_value < 0.05).sum())

def cap_counts(limit):
    subset = ubiquitous[(ubiquitous.max_plants == limit) & (ubiquitous.rule == "all_close")]
    return subset.set_index("anti_target")["n_plants_q_lt_0.05"].astype(int).to_dict()

print("structures", len(similarity))
print("morgan at 0.80", morgan)
print("consensus at 0.80", consensus)
print("unblinded max", round(float(unblinded.consensus.max()), 3))
print(separation[["anti_target", "n_close", "cv_auc", "permutation_p"]].to_string(index=False))
print(relative[["anti_target", "n_consensus_ge_0.80", "n_consensus_ge_reference_median"]].to_string(index=False))
print(potency_rules.to_string(index=False))
print(curated.to_string(index=False))

assert len(similarity) == 14956
assert morgan == {"hERG": 212, "CYP": 85, "PXR": 147}
assert consensus == {"hERG": 137, "CYP": 45, "PXR": 69}
assert intersections["hERG∩CYP"] == 23
assert intersections["hERG∩PXR"] == 20
assert intersections["CYP∩PXR"] == 9
assert intersections["all three"] == 7
assert len(pd.read_csv(PAPER / "close_compounds.csv")) == 206
assert float(unblinded.consensus.max()) < 0.50
assert supplement.groupby("anti_target").size().to_dict() == {"CYP": 16, "PXR": 11, "hERG": 51}
assert (separation.cv_auc > 0.70).all()
assert (separation.permutation_p < 0.05).all()
assert relative.set_index("anti_target")["n_consensus_ge_reference_median"].astype(int).to_dict() == {"hERG": 142, "CYP": 859, "PXR": 1643}
assert aggregation.set_index("anti_target")["n_mean_ge_0.80"].astype(int).to_dict() == {"hERG": 188, "CYP": 67, "PXR": 108}
assert isoforms.set_index("isoform")["n_close_0.80"].astype(int).to_dict() == {"CYP1A2": 7, "CYP2C9": 25, "CYP2D6": 11, "CYP3A4": 21}
assert isoforms.set_index("isoform")["n_reference"].astype(int).to_dict() == {"CYP1A2": 1412, "CYP2C9": 1285, "CYP2D6": 1493, "CYP3A4": 2335}
assert potency_rules.set_index("anti_target")["n_nearest_ge_5"].astype(int).to_dict() == {"hERG": 51, "CYP": 16, "PXR": 11}
assert potency_rules.set_index("anti_target")["n_any_neighbor_ge_5"].astype(int).to_dict() == {"hERG": 51, "CYP": 19, "PXR": 11}
assert potency_rules.set_index("anti_target")["n_flips_to_potent"].astype(int).to_dict() == {"hERG": 0, "CYP": 3, "PXR": 0}
assert plant_counts == {
    ("hERG", "all"): 566, ("CYP", "all"): 864, ("PXR", "all"): 860,
    ("hERG", "restricted"): 12, ("CYP", "restricted"): 6, ("PXR", "restricted"): 4,
}
assert cap_counts(10) == {"hERG": 8, "CYP": 3, "PXR": 7}
assert cap_counts(20) == {"hERG": 12, "CYP": 6, "PXR": 4}
assert cap_counts(50) == {"hERG": 16, "CYP": 6, "PXR": 5}
assert curated.set_index("anti_target")["n_after_excluding_artifacts"].astype(int).to_dict() == {"hERG": 113, "CYP": 40, "PXR": 65}
assert classes.curation.value_counts().to_dict() == {
    "credible_phytochemical": 170, "xenobiotic": 20, "contaminant": 8, "uncertain": 8,
}
assert int(unmapped.set_index("set").loc["unmapped", "n_names"]) == 2835
for name in (
    "figure1_within_set_diversity.png",
    "figure2_imppat_similarity.png",
    "figure3_cutoff_robustness.png",
    "figure4_close_vs_far.png",
    "figure5_plant_enrichment.png",
):
    path = PAPER / name
    assert path.exists() and path.stat().st_size > 0
print("checks passed")
